# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | _YYYY-MM-DD_ |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | _ |
| Genutzte Suchanfragen (Phase 1) | _ |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus inspizieren + Bias-Notiz

In [33]:
import requests
import time
import base64

# 1. Korrekter API-Key (clientId aus der Doku)
API_KEY = "jobboerse-jobsuche"
headers = {"X-API-Key": API_KEY}  # Header anpassen!

# 2. BASE_URL OHNE Parameter (nur Basis-Endpoint)
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service"

# 3. Suche ausführen (Parameter in der URL)
search_url = f"{BASE_URL}/pc/v4/jobs?was=Data&wo=Bremen&berufsfeld=Informatik&size=50"
response = requests.get(search_url, headers=headers)
stellen = response.json().get("stellenangebote", [])  

# 4. Details abrufen (refnr ist base64-encodiert!)
for stelle in stellen:
    refnr = base64.b64encode(stelle["refnr"].encode()).decode()  # refnr encodieren
    detail_url = f"{BASE_URL}/pc/v4/jobdetails/{refnr}"
    detail_response = requests.get(detail_url, headers=headers)
    detail_data = detail_response.json()

    # 5. Speichern (JSONL)
    with open("../daten/eigener_korpus", "a", encoding="utf-8") as f:
        f.write(f"{detail_data}\n")

    time.sleep(1)  # Rate-Limit beachten

In [28]:
print(response.status_code)

200


In [29]:
print(response.json().keys())

dict_keys(['stellenangebote', 'maxErgebnisse', 'page', 'size', 'woOutput', 'facetten'])


In [30]:
print(stellen)

[{'beruf': 'Fachinformatiker/in - Daten- und Prozessanalyse', 'titel': 'Data Engineer (m/w/d)', 'refnr': '15939-BB-633455-7878-6343-S', 'arbeitsort': {'plz': '28217', 'ort': 'Bremen', 'strasse': 'null', 'region': 'Bremen', 'land': 'Deutschland', 'koordinaten': {'lat': 53.0953969, 'lon': 8.7763193}, 'entfernung': '4'}, 'arbeitgeber': 'Rheinmetall AG', 'aktuelleVeroeffentlichungsdatum': '2026-04-23', 'modifikationsTimestamp': '2026-04-27T12:49:28.406', 'eintrittsdatum': '2026-04-23', 'kundennummerHash': 'mDtoRsc-Vw0swe56F2UrizVz-AWYNSy7i3tzAduv66s='}, {'beruf': 'Bachelor Professional - IT (Datenanalyse)', 'titel': 'Data Analyst (m/w/d) für AIRBUS', 'refnr': '10001-1003011936-S', 'arbeitsort': {'ort': 'Bremen', 'strasse': 'null', 'region': 'Bremen', 'land': 'Deutschland', 'koordinaten': {'lat': 53.11788657000005, 'lon': 8.772699361000093}, 'entfernung': '2'}, 'arbeitgeber': 'SimpleXX GmbH', 'aktuelleVeroeffentlichungsdatum': '2026-05-02', 'modifikationsTimestamp': '2026-05-02T23:00:46.488

In [31]:
print(response.text)

{
  "stellenangebote" : [ {
    "beruf" : "Fachinformatiker/in - Daten- und Prozessanalyse",
    "titel" : "Data Engineer (m/w/d)",
    "refnr" : "15939-BB-633455-7878-6343-S",
    "arbeitsort" : {
      "plz" : "28217",
      "ort" : "Bremen",
      "strasse" : "null",
      "region" : "Bremen",
      "land" : "Deutschland",
      "koordinaten" : {
        "lat" : 53.0953969,
        "lon" : 8.7763193
      },
      "entfernung" : "4"
    },
    "arbeitgeber" : "Rheinmetall AG",
    "aktuelleVeroeffentlichungsdatum" : "2026-04-23",
    "modifikationsTimestamp" : "2026-04-27T12:49:28.406",
    "eintrittsdatum" : "2026-04-23",
    "kundennummerHash" : "mDtoRsc-Vw0swe56F2UrizVz-AWYNSy7i3tzAduv66s="
  }, {
    "beruf" : "Bachelor Professional - IT (Datenanalyse)",
    "titel" : "Data Analyst (m/w/d) für AIRBUS",
    "refnr" : "10001-1003011936-S",
    "arbeitsort" : {
      "ort" : "Bremen",
      "strasse" : "null",
      "region" : "Bremen",
      "land" : "Deutschland",
      "koordina

## Phase 2 — κ-Tabelle + drei Edge Cases